# Paper 3 — Climate exposure under SSP3-7.0 multi-model ensemble

Nature Energy paper. Runs the calibrated household-energy ABM for Newcastle
across three Climate-DT realizations of SSP3-7.0 (ICON, IFS-FESOM, IFS-NEMO),
2020–2040, and reports the multi-model envelope of city-level demand.

**Climate inputs** live in the sibling `climate-to-abm` repo, schema:
`(latitude, longitude, timestamp, temp_C)`. Validated by `climate-to-abm validate`
before this notebook is run.

**Calibrated config** auto-detected from newest `results/calibration_*/`.

## Skeleton status

This is a runnable skeleton — the analysis cells (figures, exports) are scaffolded
with TODOs. The expensive parts (parallel ABM runs, ensemble aggregation) are wired.


## Imports

In [ ]:
from __future__ import annotations

from itertools import product
from pathlib import Path
import hashlib
import multiprocessing as mp
import random
import tempfile
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from household_energy.model import EnergyModel

warnings.filterwarnings('ignore', message='.*GeoSeries.notna.*')
warnings.filterwarnings('ignore', category=FutureWarning)


## 1) Settings

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

# Newcastle synthpop + sociodemographic merge (same as paper1/paper2)
GEOJSON  = ROOT / 'data' / 'epc_abm_newcastle.geojson'
HIDP_CSV = ROOT / 'data' / 'hidp_uprn_matches_tiered.csv'

# Climate-DT projections live in the sibling preprocessor repo.
# Each file: timestamp / latitude / longitude / temp_C; 2020-09-01 → 2039-12-31.
CLIMATE_REPO = Path('/Users/abeltran/Documents/GitHub/climate-to-abm/data')
CLIMATE_MODELS_FULL = {
    'icon':      CLIMATE_REPO / 'newcastle_climate_dt_ssp370_icon_2t_2020_2040.parquet',
    'ifs_fesom': CLIMATE_REPO / 'newcastle_climate_dt_ssp370_ifs_fesom_2t_2020_2040.parquet',
    'ifs_nemo':  CLIMATE_REPO / 'newcastle_climate_dt_ssp370_ifs_nemo_2t_2020_2040.parquet',
}
for _name, _path in CLIMATE_MODELS_FULL.items():
    if not _path.exists():
        raise FileNotFoundError(f'{_name}: {_path}')

OUTDIR = ROOT / 'notebooks' / 'results' / 'paper3_climate'
OUTDIR.mkdir(parents=True, exist_ok=True)

# ── Runtime ──────────────────────────────────────────────────────────────────
# True  → 28-day window for iteration (sampled at START_UTC)
# False → full ~19-year projection from data start; per-calendar-year demand
#         captured via EnergyModel.annual_kwh_by_year so the time-series figure
#         is just a groupby (no scalar collapse, no 1/20 annualisation hack).
USE_FAST    = True
WINDOW_DAYS = 28
N_PROCS     = max(1, min(8, (mp.cpu_count() or 2) - 1))

# Climate data spans 2020-09-01 → 2039-12-31 (19y 4m). Anchor START_UTC at the
# data start so the long run stays in-window (the previous 2025-01-01 +20y
# default would overrun by 5 years and rely on whatever climate-loop fallback
# the model uses). 19 years × 365 × 24 = 166,440h leaves a small tail buffer.
START_UTC = '2020-09-01T00:00:00Z'

WINDOW_HOURS  = (WINDOW_DAYS * 24) if USE_FAST else (19 * 365 * 24)
# Only used in fast mode to extrapolate a 28-day winter sample to annual.
# In full mode we read annual_kwh_by_year directly, no scaling needed.
ANNUAL_FACTOR = (365.0 / WINDOW_DAYS) if USE_FAST else 1.0

PROCESS_COLUMN = 'lsoa_code'

# ── Climate slices ───────────────────────────────────────────────────────────
# Slice each of the three model parquets to [START − 48h, START + WINDOW + 48h]
# once at notebook start. Workers point at the slices instead of re-reading the
# full multi-decade files. Fast mode: ~0.1 MB per slice vs ~20 MB full (~200×
# per-worker reduction). With 8 workers × 3 models the savings stack.
# Slice filename embeds START + WINDOW so fast and annual coexist.
_buf_hours = 48
_slice_tag = f'{pd.Timestamp(START_UTC).strftime("%Y%m%d")}_{WINDOW_HOURS}h'
CLIMATE_MODELS = {}  # name → path of sliced parquet (workers use these)

for _name, _full_path in CLIMATE_MODELS_FULL.items():
    _slice_path = OUTDIR / f'climate_slice_{_name}_{_slice_tag}.parquet'
    if not _slice_path.exists():
        print(f'  Slicing {_name}: → {_slice_path.name}', end=' ', flush=True)
        _start = pd.Timestamp(START_UTC) - pd.Timedelta(hours=_buf_hours)
        _end   = pd.Timestamp(START_UTC) + pd.Timedelta(hours=WINDOW_HOURS + _buf_hours)
        _full  = pd.read_parquet(_full_path)
        _mask  = (_full['timestamp'] >= _start) & (_full['timestamp'] <= _end)
        _full.loc[_mask].reset_index(drop=True).to_parquet(_slice_path, index=False)
        del _full
        print('done.')
    CLIMATE_MODELS[_name] = _slice_path

# ── Calibration auto-pickup ──────────────────────────────────────────────────
_cal_dirs = sorted((ROOT / 'results').glob('calibration_*'), key=lambda p: p.name)
if not _cal_dirs:
    raise FileNotFoundError(f"No calibration_* directory under {ROOT / 'results'}")
CAL_DIR = _cal_dirs[-1]
CAL_YAML = CAL_DIR / 'calibrated_config.yaml'
if not CAL_YAML.exists():
    raise FileNotFoundError(f"Calibrated config not found: {CAL_YAML}")
with CAL_YAML.open() as _f:
    CALIBRATED_CFG: dict = yaml.safe_load(_f) or {}
print(f'Calibration base:  {CAL_DIR.name}')


def _deep_merge(base: dict, override: dict) -> dict:
    out = dict(base)
    for k, v in (override or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge(out[k], v)
        else:
            out[k] = v
    return out


# ── Aesthetics ────────────────────────────────────────────────────────────────
MODEL_COLORS = {'icon': '#4c78a8', 'ifs_fesom': '#54a24b', 'ifs_nemo': '#f58518'}
MODEL_LABELS = {'icon': 'ICON', 'ifs_fesom': 'IFS-FESOM', 'ifs_nemo': 'IFS-NEMO'}
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
                     'axes.spines.top': False, 'axes.spines.right': False})

print(f'Models:            {list(CLIMATE_MODELS.keys())}')
print(f'Workers:           {N_PROCS} (maxtasksperchild=1)')
print(f'Window:            {WINDOW_HOURS:,}h  '
      f'({"FAST 28-day" if USE_FAST else "FULL ~19-year"})')
print(f'Window start:      {START_UTC}')
print(f'Annual factor:     {ANNUAL_FACTOR:.3f} (applied only in fast mode)')
print(f'Output:            {OUTDIR}')

# Audit: most-impactful calibrated parameters in the run log
_audit_keys = [
    'heating_setpoint_C', 'heating_slope_kWh_per_deg',
    'baseline_anchor_elec_kwh_per_hour', 'baseline_anchor_gas_kwh_per_hour',
]
print('Calibrated values (will be used by every worker):')
for k in _audit_keys:
    v = CALIBRATED_CFG.get('model', {}).get(k)
    if v is not None:
        print(f'  {k:42s} {v}')


## 2) Load and shard the synthetic population

Same enrichment as paper1/paper2 — EPC GeoJSON merged with HIDP sociodemographic
columns. LSOA-sharded parquets enable parallel execution across ~185 shards.

In [ ]:
def load_enriched_gdf(geojson_path: Path, hidp_csv_path: Path | None) -> gpd.GeoDataFrame:
    g = gpd.read_file(geojson_path)
    g['UPRN'] = g['UPRN'].astype(str).str.strip()
    if hidp_csv_path and hidp_csv_path.exists():
        hidp = pd.read_csv(hidp_csv_path, low_memory=False)
        hidp.columns = [c.strip() for c in hidp.columns]
        hidp['uprn_chr'] = hidp['uprn_chr'].astype(str).str.strip()
        hidp = hidp.drop_duplicates(subset=['uprn_chr'])
        g = g.merge(hidp, how='left', left_on='UPRN', right_on='uprn_chr',
                    suffixes=('_geo', '_hidp'))
        for base in ['lsoa_code', 'ward_code']:
            geo_col, hidp_col = f'{base}_geo', f'{base}_hidp'
            if base not in g.columns:
                if geo_col in g.columns and hidp_col in g.columns:
                    g[base] = g[geo_col].combine_first(g[hidp_col])
                elif geo_col in g.columns:
                    g[base] = g[geo_col]
                elif hidp_col in g.columns:
                    g[base] = g[hidp_col]
    return g


def to_wgs84_points(gdf_in: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    g = gdf_in.dropna(subset=['geometry']).copy()
    g = g.to_crs(4326) if g.crs else g.set_crs(4326)
    if (g.geometry.geom_type != 'Point').any():
        g['geometry'] = g.geometry.centroid
    return g


gdf_all = to_wgs84_points(load_enriched_gdf(GEOJSON, HIDP_CSV if HIDP_CSV.exists() else None))
gdf_all['AgentID'] = gdf_all['UPRN'].astype(str)

LSOA_UNITS = (
    gdf_all[PROCESS_COLUMN].astype(str).replace('', np.nan).dropna().unique().tolist()
)

UNIT_INPUT_DIR = OUTDIR / 'unit_inputs'
UNIT_INPUT_DIR.mkdir(exist_ok=True)
written = 0
for unit in LSOA_UNITS:
    fpath = UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit}.parquet'
    if not fpath.exists():
        gdf_all[gdf_all[PROCESS_COLUMN].astype(str) == unit].to_parquet(fpath, index=False)
        written += 1

print(f'City:              {len(gdf_all):,} dwellings | {len(LSOA_UNITS)} LSOAs')
print(f'Parquets:          {written} newly written')


## 3) Model runner

`_run_lsoa_model` runs the calibrated ABM for one (LSOA × climate-model) cell,
returning annualized city-share demand. Seeded deterministically across kernel
restarts via sha256.

No policy intervention — Paper 3 is exposure of the **existing** stock to
climate uncertainty, so adoption is held at zero.

In [ ]:
def _build_and_run_yearly(gdf_in: gpd.GeoDataFrame,
                          climate_parquet: Path,
                          cfg_dict: dict | None) -> dict[int, float]:
    """Run EnergyModel for WINDOW_HOURS; return {calendar_year: lsoa_kwh}.

    Replaces the prior scalar-summing version. Reads ``annual_kwh_by_year`` off
    every household agent and aggregates per calendar year across the LSOA.
    In fast mode this yields a single key (the year containing START_UTC); in
    full mode it yields ~19 keys, which the downstream code unwinds into a
    (unit, model, year) tidy frame.
    """
    merged = _deep_merge(CALIBRATED_CFG, cfg_dict or {})
    with tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False) as tmp:
        yaml.safe_dump(merged, tmp)
        cfg_path = tmp.name

    m = EnergyModel(
        gdf=gdf_in,
        climate_parquet=str(climate_parquet),
        climate_start=START_UTC,
        collect_agent_level=False,
        agent_collect_every=168,
        config_path=cfg_path,
    )
    for _ in range(WINDOW_HOURS):
        m.step()

    by_year_lsoa: dict[int, float] = {}
    for h in m.household_agents:
        for yr, kwh in (getattr(h, 'annual_kwh_by_year', {}) or {}).items():
            try:
                ykey = int(yr)
            except (TypeError, ValueError):
                continue
            by_year_lsoa[ykey] = by_year_lsoa.get(ykey, 0.0) + float(kwh)
    return by_year_lsoa


def _run_lsoa_model(unit_val: str, model_name: str) -> list[dict]:
    """One (LSOA, model) job. Returns a list of rows, one per calendar year."""
    g = gpd.read_parquet(UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit_val}.parquet')
    g['AgentID'] = g['UPRN'].astype(str)

    seed_key = f'paper3|{unit_val}|{model_name}|{START_UTC}|{WINDOW_HOURS}'.encode()
    seed = int.from_bytes(hashlib.sha256(seed_key).digest()[:4], 'big')
    np.random.seed(seed); random.seed(seed)

    cfg = {'meta': {'name': f'paper3_{model_name}'},
           'model': {'heatpump_adoption_rate': 0.0}}
    by_year = _build_and_run_yearly(g, CLIMATE_MODELS[model_name], cfg)

    if not by_year:
        # Shouldn't happen, but guard against returning empty list (breaks
        # the partial-cache "skip if shard exists" logic downstream).
        return [{'unit': unit_val, 'model': model_name,
                 'year': pd.Timestamp(START_UTC).year, 'kwh': 0.0}]

    return [
        {'unit': unit_val, 'model': model_name,
         'year': yr, 'kwh': kwh * ANNUAL_FACTOR}  # ANNUAL_FACTOR=1.0 in full mode
        for yr, kwh in sorted(by_year.items())
    ]


def _run_lsoa_model_star(args):
    return _run_lsoa_model(*args)


def _run_parallel(jobs, n_procs: int):
    """Fork-based pool, recycling workers after each job (maxtasksperchild=1)
    so per-run climate-dataframe allocations release back to the OS instead
    of accumulating across the sweep. Same fix as sensitivity_analysis.
    """
    if n_procs <= 1 or not jobs:
        return [_run_lsoa_model_star(j) for j in jobs]
    try:
        ctx = mp.get_context('fork')
        with ctx.Pool(processes=n_procs, maxtasksperchild=1) as pool:
            return pool.map(_run_lsoa_model_star, jobs)
    except Exception as e:
        print(f'Parallel pool failed ({e}); falling back to serial.')
        return [_run_lsoa_model_star(j) for j in jobs]


## 4) Run the three-model ensemble

One (LSOA × model) job per pool worker. Results cached to parquet; re-running
the cell skips any (LSOA, model) combination already present.

In [ ]:
# Window-tagged ensemble cache so fast and full results coexist without
# overwriting each other. Schema is now tidy: (unit, model, year, kwh) — one
# row per LSOA-model-calendar-year. Fast mode yields 1 row per (LSOA, model);
# full mode yields ~19 (one per calendar year in window).
ensemble_cache = OUTDIR / f'ensemble_raw_{WINDOW_HOURS}h.parquet'

# Per-shard cache directory so a long annual run that dies at LSOA 170/185
# only loses the in-flight shard. Each (unit, model) writes one shard parquet
# of 1–19 rows; re-running picks up missing combinations.
SHARD_DIR = OUTDIR / f'shards_{WINDOW_HOURS}h'
SHARD_DIR.mkdir(exist_ok=True)


def _shard_path(unit_val: str, model_name: str) -> Path:
    return SHARD_DIR / f'{model_name}__{PROCESS_COLUMN}={unit_val}.parquet'


def _run_shard_cached(unit_val: str, model_name: str) -> pd.DataFrame:
    p = _shard_path(unit_val, model_name)
    if p.exists():
        return pd.read_parquet(p)
    rows = _run_lsoa_model(unit_val, model_name)
    df = pd.DataFrame(rows)
    df.to_parquet(p, index=False)
    return df


def _run_shard_cached_star(args):
    return _run_shard_cached(*args)


all_jobs = [(unit, model) for unit in LSOA_UNITS for model in CLIMATE_MODELS.keys()]
missing  = [(u, m) for u, m in all_jobs if not _shard_path(u, m).exists()]

print(f'Total jobs:       {len(all_jobs):,}  ({len(LSOA_UNITS)} LSOAs × {len(CLIMATE_MODELS)} models)')
print(f'Cached on disk:   {len(all_jobs) - len(missing):,}')
print(f'To run:           {len(missing):,}')

if missing:
    print(f'Running {len(missing):,} shards across {N_PROCS} workers '
          f'(maxtasksperchild=1)...')
    if N_PROCS > 1:
        try:
            ctx = mp.get_context('fork')
            with ctx.Pool(processes=N_PROCS, maxtasksperchild=1) as pool:
                pool.map(_run_shard_cached_star, missing)
        except Exception as e:
            print(f'Parallel pool failed ({e}); falling back to serial.')
            for j in missing:
                _run_shard_cached_star(j)
    else:
        for j in missing:
            _run_shard_cached_star(j)

# Materialise the full tidy frame from all shards (always reads from disk so
# the in-memory state matches what's persisted)
ensemble_raw = pd.concat(
    [pd.read_parquet(_shard_path(u, m)) for u, m in all_jobs],
    ignore_index=True,
)
ensemble_raw.to_parquet(ensemble_cache, index=False)
print(f'Saved ensemble ({len(ensemble_raw):,} rows) → {ensemble_cache.name}')

# City-by-year aggregation: sum LSOAs within each (model, year)
city_year = (
    ensemble_raw
    .groupby(['model', 'year'], as_index=False)
    .agg(city_kwh=('kwh', 'sum'), n_lsoas=('unit', 'nunique'))
)
city_year['city_gwh'] = city_year['city_kwh'] / 1e6
city_year.to_csv(OUTDIR / f'ensemble_city_by_year_{WINDOW_HOURS}h.csv', index=False)

print('\nCity-level (model × year), GWh:')
print(
    city_year.pivot_table(index='year', columns='model', values='city_gwh')
             .round(2).to_string()
)


## 5) Figure — multi-model demand envelope

Headline figure for Paper 3. The figure type adapts to the data on hand:

- **Full annual mode** (USE_FAST=False): time-series line plot showing each model's
  citywide annual demand by calendar year, with the ensemble mean and ±1σ band overlaid.
  Partial-year buckets at the leading and trailing edges of the window are auto-trimmed
  using a coverage heuristic (drops years whose median GWh is <80% of the median across
  full years).

- **Fast mode** (USE_FAST=True): bar chart of the three models' single-year demand,
  scaled from the 28-day sample by ANNUAL_FACTOR. Mean and ±1σ shown as horizontal
  reference lines. Use this for iteration; the time-series version is the publishable one.

The figure auto-adapts based on `n_years` in the city_year frame, so flipping `USE_FAST`
swaps the figure shape without any code change.

In [ ]:
# Picks a time-series figure when the data covers >1 year, else a bar chart.
# Both render from the same (model, year, city_gwh) frame so flipping
# USE_FAST changes the figure shape without code changes.

order = list(CLIMATE_MODELS.keys())
n_years = city_year['year'].nunique()

if n_years > 1:
    # ── Time-series envelope (full annual mode) ───────────────────────────────
    # Drop leading and trailing partial-year buckets that exist because the
    # window doesn't align with calendar-year boundaries. Keeps only years
    # where every model has hour-coverage ≥ 350 days as a proxy for "complete".
    HOURS_PER_FULL_YEAR = 350 * 24
    hours_per_year = (
        ensemble_raw.groupby('year')['kwh'].count()
        .div(len(LSOA_UNITS) * len(CLIMATE_MODELS))
    )
    # We don't have direct hour-count here (kwh is summed kWh, not a count).
    # Use a simpler heuristic: drop years that are <80% of the median annual
    # kwh across models (these are partial-year buckets).
    yr_med = city_year.groupby('year')['city_gwh'].mean()
    keep_years = yr_med[yr_med >= 0.80 * yr_med.median()].index.tolist()
    plot_df = city_year[city_year['year'].isin(keep_years)].copy()

    fig, ax = plt.subplots(figsize=(9, 4.8))
    for m in order:
        sub = plot_df[plot_df['model'] == m].sort_values('year')
        ax.plot(sub['year'], sub['city_gwh'], color=MODEL_COLORS[m],
                marker='o', ms=4, lw=1.6, label=MODEL_LABELS[m])

    # Ensemble mean and ±1σ band across models, year-by-year
    mean_by_yr = plot_df.groupby('year')['city_gwh'].mean()
    std_by_yr  = plot_df.groupby('year')['city_gwh'].std(ddof=1)
    ax.plot(mean_by_yr.index, mean_by_yr.values, color='#444', lw=1.2, ls='--',
            label='Ensemble mean', zorder=1)
    ax.fill_between(mean_by_yr.index,
                    mean_by_yr - std_by_yr, mean_by_yr + std_by_yr,
                    color='#888', alpha=0.18, label='±1σ across models', zorder=0)

    ax.set_xlabel('Calendar year')
    ax.set_ylabel('City annual demand (GWh/yr)')
    ax.set_title('Newcastle demand under SSP3-7.0 — multi-model time-series',
                 fontsize=11, fontweight='bold')
    ax.legend(frameon=False, fontsize=9, ncol=2, loc='upper right')
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    fig.savefig(OUTDIR / f'fig1_ensemble_timeseries_{WINDOW_HOURS}h.png')
    plt.show()
    print(f'Saved fig1_ensemble_timeseries_{WINDOW_HOURS}h.png  '
          f'({len(keep_years)} full years, '
          f'{len(city_year["year"].unique()) - len(keep_years)} partial years trimmed)')

else:
    # ── Bar chart (fast 28-day mode, single year) ─────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    vals = [float(city_year.loc[city_year['model'] == m, 'city_gwh'].iloc[0]) for m in order]
    colors = [MODEL_COLORS[m] for m in order]

    bars = ax.bar(range(len(order)), vals, color=colors, width=0.6, edgecolor='white')
    mean = float(np.mean(vals))
    std  = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    ax.axhline(mean, ls='--', color='#444', lw=1.0,
               label=f'Ensemble mean ({mean:.1f} GWh/yr)')
    ax.axhspan(mean - std, mean + std, color='#999', alpha=0.15,
               label=f'±1σ ({std:.2f} GWh/yr)')

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.01,
                f'{v:.1f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([MODEL_LABELS[m] for m in order])
    ax.set_ylabel('City annual demand (GWh/yr)')
    ax.set_title(f'Newcastle demand under SSP3-7.0 — three-model ensemble\n'
                 f'(fast 28-day sample, scaled ×{ANNUAL_FACTOR:.2f}; start {START_UTC})',
                 fontsize=10)
    ax.legend(frameon=False, fontsize=9)
    ax.set_ylim(0, max(vals) * 1.18)
    fig.tight_layout()
    fig.savefig(OUTDIR / f'fig1_ensemble_bar_{WINDOW_HOURS}h.png')
    plt.show()
    print(f'Saved fig1_ensemble_bar_{WINDOW_HOURS}h.png')


## 6) Exposure distribution (TODO)

Per-dwelling distribution of climate-driven demand change under each model,
broken out by income quintile and EPC band. This is the equity dimension of the
exposure story — who is most exposed to the warmest/coolest realizations.

Inputs needed:
- Agent-level kwh output (set `collect_agent_level=True` in `_build_and_run` or
  add a second pass that captures per-agent annuals)
- Merge against `hh_income_band` and `sap_band_ord` from `gdf_all`


In [ ]:
# TODO: agent-level exposure distribution
# Requires re-running with collect_agent_level=True or post-hoc capture of
# household_agents' annual_kwh_by_year inside _build_and_run.


## 7) Export

In [ ]:
print('Outputs:')
for f in sorted(OUTDIR.glob('*.csv')):
    print(f'  {f.name}')
for f in sorted(OUTDIR.glob('*.png')):
    print(f'  {f.name}')
